# Test de `get_fundamentals`

Comprueba que `DataProviderImpl.get_fundamentals` funciona correctamente para 3 tickers (incluyendo `ITX.MC`), y pivota el resultado (formato largo: `ticker`, `year`, `metric`, `value`) a formato ancho: una fila por `ticker`+`year`, una columna por métrica.

In [3]:
import sys
from pathlib import Path

# Este notebook vive en notebooks/, así que src/ está en el directorio padre.
SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd

from providers.data_provider_impl import DataProviderImpl

pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [4]:
tickers = ["ITX.MC"]

provider = DataProviderImpl()
fundamentals = provider.get_fundamentals(tickers)
fundamentals = fundamentals[fundamentals["ticker"].isin(tickers)]
fundamentals

,ticker,year,metric,value
512,ITX.MC,2026,EBITDA,11545000000.00
513,ITX.MC,2026,Cost Of Revenue,16642000000.00
514,ITX.MC,2026,Total Revenue,39864000000.00
515,ITX.MC,2025,EBITDA,10946000000.00
516,ITX.MC,2025,Cost Of Revenue,16288000000.00
517,ITX.MC,2025,Total Revenue,38632000000.00
518,ITX.MC,2024,EBITDA,10038000000.00
519,ITX.MC,2024,Cost Of Revenue,15186000000.00
520,ITX.MC,2024,Total Revenue,35947000000.00
521,ITX.MC,2023,EBITDA,8278000000.00


In [5]:
fundamentals["metric"].nunique()

7

## Pivot: una fila por ticker/año, columnas = métricas

In [6]:
pivoted = (
    fundamentals
    .pivot_table(index=["ticker", "year"], columns="metric", values="value", aggfunc="first")
    .reset_index()
    .sort_values(["ticker", "year"], ascending=[True, False])
    .reset_index(drop=True)
)
pivoted.columns.name = None
pivoted

,ticker,year,Cost Of Revenue,EBITDA,Free Cash Flow,Total Revenue,currentRatio,debtToEquity,operatingMargins
0,ITX.MC,2026,16642000000.00,11545000000.00,6520000000.00,39864000000.00,1.54,0.30,0.20
1,ITX.MC,2025,16288000000.00,10946000000.00,6616000000.00,38632000000.00,1.61,0.30,0.20
2,ITX.MC,2024,15186000000.00,10038000000.00,6795000000.00,35947000000.00,1.79,0.31,0.19
3,ITX.MC,2023,14011000000.00,8278000000.00,5259000000.00,32569000000.00,1.80,0.33,0.18
